##### File Name: firedetection.ipynb
##### Project Name: Deep Learning Model for Detection of Forest Fires
##### Project Author: Spencer Baysinger
##### Project Details:  
  - This project utilized IEEE Flame Datasets, Python packages such as: NumPy, Pandas, and TensorFlow to 
  - build a convolution neural network that classifies fires using a 256x256 neural matrix with a binary 
  - output of fire/no-fire classification
  - (IEEE "The Flame Dataset")[https://ieee-dataport.org/open-access/flame-dataset-aerial-imagery-pile-burn-detection-using-drones-uavs]
    7) Training_Validation images for Fire_vs_NoFire image classification	1.18 GB
    8) Test images for Fire_vs_NoFire image classification	287.58 MB

---------------

#### Version History

Version 0.3
  - Make model2
  - Train model2
  - Analyze results

Version 0.2
  - Make a hypothesis about the results
  - Make a plan for additional models to improve accuracy and loss

Version 0.1
  - Build model 1
  - Analyze results

Version 0.0
  - Writing Intro
  - Setting up data frames

In [23]:
# ------------------------------
# Import the necessary libraries
# ------------------------------

import tensorflow as tf
import time
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.callbacks import ReduceLROnPlateau

# -----------------------------
# Locate dataset and process it
# -----------------------------

train_dir = r'C:\Users\TheSp\Downloads\IEEE Flame\Training'
  # Total images = 39,375
  # No Fire Training = train_dir + '\No_Fire'
  # 14,357 images
  # Fire Training = train_dir + '\Fire'
  # 25,018 images
test_dir = r'C:\Users\TheSp\Downloads\IEEE Flame\Test'
  # Total images = 8,617
  # No Fire Test = test_dir + '\No_Fire'
  # 3,480 images
  # Fire Test = test_dir + '\Fire'
  # 5,137 images

# --------------------------------
# Image data generators for model1
# --------------------------------

# Image data generator for training with data augmentation
train_datagen_1 = ImageDataGenerator(rescale=1./255)

# Image data generator for validation and test (no data augmentation, only rescaling)
test_datagen_1 = ImageDataGenerator(rescale=1./255)

# Load and preprocess the training data
train_generator_1 = train_datagen_1.flow_from_directory(
  train_dir,
  target_size=(254, 254),
  batch_size=32,
  class_mode='binary'
)

# Load and preprocess the test data
test_generator_1 = test_datagen_1.flow_from_directory(
  test_dir,
  target_size=(254, 254),
  batch_size=32,
  class_mode='binary'
)

# --------------------------------
# Image data generators for model2
# --------------------------------

# Image data generator for training with data augmentation
train_datagen_2 = ImageDataGenerator(
    rescale=1./255,
    rotation_range=50,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Image data generator for validation and test (no data augmentation, only rescaling)
test_datagen_2 = ImageDataGenerator(rescale=1./255)

# Load and preprocess the training data
train_generator_2 = train_datagen_2.flow_from_directory(
  train_dir,
  target_size=(254, 254),
  batch_size=32,
  class_mode='binary'
)

# Load and preprocess the test data
test_generator_2 = test_datagen_2.flow_from_directory(
  test_dir,
  target_size=(254, 254),
  batch_size=32,
  class_mode='binary'
)

Found 39375 images belonging to 2 classes.
Found 8617 images belonging to 2 classes.
Found 39375 images belonging to 2 classes.
Found 8617 images belonging to 2 classes.


In [10]:
#---------------------
# Build the CNN model1
#---------------------

model1 = Sequential([
  Conv2D(32, (3, 3), activation='relu', input_shape=(254, 254, 3)),
  MaxPooling2D((2, 2)),
  Conv2D(64, (3, 3), activation='relu'),
  MaxPooling2D((2, 2)),
  Flatten(),
  Dense(128, activation='relu'),
  Dropout(0.5),
  Dense(1, activation='sigmoid')
])

#--------------
# Model Details
#--------------
  # Input Layer:
    # Accepts images of size 254x254 with 3 channels (RGB).
    # The shape is defined as input_shape=(254, 254, 3).

  # Convolutional Layer 1:
    # 32 filters with a kernel size of 3x3.
    # ReLU activation introduces non-linearity to the model.
    # Outputs a feature map.

  # MaxPooling Layer 1:
    # Reduces spatial dimensions by half (e.g., 254x254 becomes 127x127).

  # Convolutional Layer 2:
    # 64 filters with a kernel size of 3x3.
    # Extracts more complex features from the reduced feature map.

  # MaxPooling Layer 2:
    # Again reduces spatial dimensions by half (e.g., 127x127 becomes 63x63).

  # Flatten Layer:
    # Converts the 2D feature maps into a 1D vector for input into dense layers.

  # Dense Layer:
    # Fully connected layer with 128 neurons and ReLU activation.
    # Learns high-level features.

  # Dropout Layer:
    # Drops 50% of neurons during training to prevent overfitting.

  # Output Layer:
    # Single neuron with sigmoid activation for binary classification.
    # Outputs a probability between 0 and 1 (e.g., 1 for "Fire," 0 for "No Fire").

c:\Users\TheSp\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:99: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


In [ ]:
# ---------------
# Compile Model 1
# ---------------

# Compile Model
model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model1 summary
model1.summary()

# Train model1
start_time = time.time()
history = model1.fit(
    train_generator_1,
    steps_per_epoch=train_generator_1.samples // train_generator_1.batch_size,
    epochs=10,  # Adjust epochs as needed
    validation_data=test_generator_1,
    validation_steps=test_generator_1.samples // test_generator_1.batch_size
)
end_time = time.time()

# Evaluate model1 on the test set
test_loss, test_accuracy = model1.evaluate(test_generator_1)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Calculate compile time
total_seconds = end_time - start_time
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

# Print compile time in minutes and seconds
print(f"Model compile time: {minutes} minutes, {seconds} seconds")

# Save model1 for future use
model1.save('my_model1.keras')

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 252, 252, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 126, 126, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 124, 124, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 246016)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │    31,490,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,509,697 (120.20 MB)

 Trainable params: 31,509,697 (120.20 MB)

 Non-trainable params: 0 (0.00 B)

...

c:\Users\TheSp\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1230/1230 ━━━━━━━━━━━━━━━━━━━━ 333s 270ms/step - accuracy: 0.9436 - loss: 0.2094 - val_accuracy: 0.4941 - val_loss: 3.6583
Epoch 2/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 26us/step - accuracy: 1.0000 - loss: 0.0044 - val_accuracy: 0.2222 - val_loss: 3.3253
Epoch 3/10


c:\Users\TheSp\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)


1230/1230 ━━━━━━━━━━━━━━━━━━━━ 329s 267ms/step - accuracy: 0.9954 - loss: 0.0133 - val_accuracy: 0.3841 - val_loss: 6.2390
Epoch 4/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 20us/step - accuracy: 1.0000 - loss: 0.0026 - val_accuracy: 0.5556 - val_loss: 1.2574
Epoch 5/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 330s 268ms/step - accuracy: 0.9974 - loss: 0.0102 - val_accuracy: 0.5229 - val_loss: 2.4666
Epoch 6/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 20us/step - accuracy: 1.0000 - loss: 6.9635e-04 - val_accuracy: 0.4444 - val_loss: 0.8358
Epoch 7/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 329s 267ms/step - accuracy: 0.9946 - loss: 0.0223 - val_accuracy: 0.4815 - val_loss: 1.9387
Epoch 8/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 21us/step - accuracy: 1.0000 - loss: 0.0073 - val_accuracy: 0.4444 - val_loss: 1.3807
Epoch 9/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 330s 268ms/step - accuracy: 0.9975 - loss: 0.0097 - val_accuracy: 0.4845 - val_loss: 2.8856
Epoch 10/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 20us/step - accuracy: 1.0000 - l

Test Loss: 2.8972, Test Accuracy: 0.4842
Model compile time: 27 minutes, 31 seconds


Epoch 1/10
c:\Users\TheSp\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 333s 270ms/step - accuracy: 0.9436 - loss: 0.2094 - val_accuracy: 0.4941 - val_loss: 3.6583
Epoch 2/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 26us/step - accuracy: 1.0000 - loss: 0.0044 - val_accuracy: 0.2222 - val_loss: 3.3253
Epoch 3/10
c:\Users\TheSp\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 329s 267ms/step - accuracy: 0.9954 - loss: 0.0133 - val_accuracy: 0.3841 - val_loss: 6.2390
Epoch 4/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 20us/step - accuracy: 1.0000 - loss: 0.0026 - val_accuracy: 0.5556 - val_loss: 1.2574
Epoch 5/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 330s 268ms/step - accuracy: 0.9974 - loss: 0.0102 - val_accuracy: 0.5229 - val_loss: 2.4666
Epoch 6/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 20us/step - accuracy: 1.0000 - loss: 6.9635e-04 - val_accuracy: 0.4444 - val_loss: 0.8358
Epoch 7/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 329s 267ms/step - accuracy: 0.9946 - loss: 0.0223 - val_accuracy: 0.4815 - val_loss: 1.9387
Epoch 8/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 21us/step - accuracy: 1.0000 - loss: 0.0073 - val_accuracy: 0.4444 - val_loss: 1.3807
Epoch 9/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 330s 268ms/step - accuracy: 0.9975 - loss: 0.0097 - val_accuracy: 0.4845 - val_loss: 2.8856
Epoch 10/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 20us/step - accuracy: 1.0000 - loss: 2.2207e-05 - val_accuracy: 0.5556 - val_loss: 1.9120
270/270 ━━━━━━━━━━━━━━━━━━━━ 14s 51ms/step - accuracy: 0.4791 - loss: 2.8786
Test Loss: 2.8972, Test Accuracy: 0.4842
Model compile time: 27 minutes, 31 seconds

Above we see the results of Model 1, which I will use as a baseline for further research. The CNN has two convolutional layers, ReLU activation, and dropout on the dense layer. We see a drop in accuracy from (training ~99%) to (~47%) which is quite a big drop.

We can hypothesize that this drop is a result of overfitting. The model may have too many parameters, insufficient regularization from dropout or weight decay. Or a possible imbalance or quality issue in the training dataset leading to a model bias.

So for the next CNN model, we will adapt the layers to attempt to improve the model accuracy with the validation set.

In [ ]:
#---------------------
# Build the CNN model2
#---------------------

model2 = Sequential([
    Conv2D(16, (3, 3), activation='relu', kernel_regularizer=l2(0.001), input_shape=(254, 254, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(32, (3, 3), activation='relu', kernel_regularizer=l2(0.001)),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.6),
    Dense(1, activation='sigmoid')
])

class_weights = {0: 1.0, 1: 1.5}  # Adjust based on the dataset
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Learning Rate Scheduler
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, verbose=1)

#--------------
# Model2 Changes
#--------------
  # Model Changes from model1:
    # Add L2 Regularization (Weight Decay) to the Dense and Conv2D layers to penalize overly large weights:
    # Increase dropout to 0.6 to prevent overfitting.
    # Use fewer filters in the Conv2D layers to reduce model complexity.
    # Apply stronger data augmentation to simulate real-world scenarios and prevent overfitting
    # Use class weights to balance learning
    # Use Learning Rate Scheduler to reduce the learning rate during training to stabilize improvements
    # Use Early Stopping to monitor validation loss and stop training when it stops improving

c:\Users\TheSp\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:99: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


In [ ]:
# ---------------
# Compile Model 2
# ---------------

# Compile Model2
model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model2 summary
model2.summary()

# Train model2
start_time = time.time()
history_2 = model2.fit(
    train_generator_2, 
    validation_data=test_generator_2, 
    epochs=10, 
    callbacks=[early_stopping, lr_scheduler], 
    class_weight=class_weights, 
    steps_per_epoch=train_generator_2.samples // train_generator_2.batch_size, 
    validation_steps=test_generator_2.samples // test_generator_2.batch_size
)
end_time = time.time()

# Calculate compile time
total_seconds = end_time - start_time
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

# Print compile time in minutes and seconds
print(f"Model compile time: {minutes} minutes, {seconds} seconds")

# Evaluate model2 on the test set
test_loss, test_accuracy = model2.evaluate(test_generator_2)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Save model2 for future use
model2.save('my_model2.keras')

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 252, 252, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 126, 126, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 124, 124, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 123008)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │    15,745,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,750,369 (60.08 MB)

 Trainable params: 15,750,369 (60.08 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 590s 477ms/step - accuracy: 0.9393 - loss: 0.3240 - val_accuracy: 0.5859 - val_loss: 0.8954 - learning_rate: 0.0010
Epoch 2/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 48us/step - accuracy: 0.9688 - loss: 0.1501 - val_accuracy: 0.4444 - val_loss: 0.7088 - learning_rate: 0.0010
Epoch 3/10


c:\Users\TheSp\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)


1230/1230 ━━━━━━━━━━━━━━━━━━━━ 394s 319ms/step - accuracy: 0.9423 - loss: 0.3328 - val_accuracy: 0.4589 - val_loss: 2.1402 - learning_rate: 0.0010
Epoch 4/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 32us/step - accuracy: 1.0000 - loss: 0.0893 - val_accuracy: 0.6667 - val_loss: 0.5792 - learning_rate: 0.0010
Epoch 5/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 426s 346ms/step - accuracy: 0.9436 - loss: 0.3194 - val_accuracy: 0.5108 - val_loss: 1.3517 - learning_rate: 0.0010
Model compile time: 23 minutes, 31 seconds
270/270 ━━━━━━━━━━━━━━━━━━━━ 12s 44ms/step - accuracy: 0.5859 - loss: 0.8895


Test Loss: 0.8966, Test Accuracy: 0.5857


Epoch 1/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 590s 477ms/step - accuracy: 0.9393 - loss: 0.3240 - val_accuracy: 0.5859 - val_loss: 0.8954 - learning_rate: 0.0010
Epoch 2/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 48us/step - accuracy: 0.9688 - loss: 0.1501 - val_accuracy: 0.4444 - val_loss: 0.7088 - learning_rate: 0.0010
Epoch 3/10
c:\Users\TheSp\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 394s 319ms/step - accuracy: 0.9423 - loss: 0.3328 - val_accuracy: 0.4589 - val_loss: 2.1402 - learning_rate: 0.0010
Epoch 4/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 32us/step - accuracy: 1.0000 - loss: 0.0893 - val_accuracy: 0.6667 - val_loss: 0.5792 - learning_rate: 0.0010
Epoch 5/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 426s 346ms/step - accuracy: 0.9436 - loss: 0.3194 - val_accuracy: 0.5108 - val_loss: 1.3517 - learning_rate: 0.0010
Model compile time: 23 minutes, 31 seconds
270/270 ━━━━━━━━━━━━━━━━━━━━ 12s 44ms/step - accuracy: 0.5859 - loss: 0.8895
Test Loss: 0.8966, Test Accuracy: 0.5857

So the results above were pretty interesting blah blah, now we'll try using a pretrained model ResNet using datagen data from model1 

In [ ]:
# Import ResNet Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

# --------------------------------
# Image data generators for model3
# --------------------------------

# Image data generator for training with data augmentation
train_datagen_3 = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ResNet-specific preprocessing
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Image data generator for validation and test (no data augmentation, only rescaling)
test_datagen_3 = ImageDataGenerator(preprocessing_function=preprocess_input)

# Load the training data
train_generator_3 = train_datagen_3.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # Resize images to 224x224
    batch_size=32,
    class_mode='binary',
)

# Load the test data
test_generator_3 = test_datagen_3.flow_from_directory(
    test_dir,
    target_size=(224, 224),  # Resize images to 224x224
    batch_size=32,
    class_mode='binary',
)

#-------------
# Build Model3
#-------------

# Load the base ResNet model
base_model = ResNet50(
    include_top=False,  # Exclude the top fully connected layer
    weights='imagenet',  # Load pretrained weights
    input_shape=(224, 224, 3)  # Input shape for ResNet
)

# Freeze the base model to retain pretrained weights
base_model.trainable = False

# Build the model
model3 = Sequential([
    base_model,  # Add the pre-trained ResNet as the base
    GlobalAveragePooling2D(),  # Pool the feature maps into a single vector
    Dense(128, activation='relu'),  # Fully connected layer
    Dropout(0.5),  # Dropout for regularization
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
start_time = time.time()
history = model3.fit(
    train_generator_3,
    steps_per_epoch=train_generator_3.samples // train_generator_3.batch_size,
    epochs=5,
    validation_data=test_generator_3,
    validation_steps=test_generator_3.samples // test_generator_3.batch_size,
    validation_freq=2,  # Validate every 2 epochs
    callbacks=[early_stopping, lr_scheduler],
)
end_time = time.time()

# Calculate compile time
total_seconds = end_time - start_time
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

# Print compile time in minutes and seconds
print(f"Model compile time: {minutes} minutes, {seconds} seconds")

# Evaluate model3 on the test set
test_loss, test_accuracy = model2.evaluate(test_generator_2)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Save model2 for future use
model3.save('my_model2.keras')


Found 39375 images belonging to 2 classes.
Found 8617 images belonging to 2 classes.
Epoch 1/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 767s 618ms/step - accuracy: 0.9599 - loss: 0.1104 - val_accuracy: 0.6442 - val_loss: 2.2633 - learning_rate: 0.0010
Epoch 2/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 1s 271us/step - accuracy: 1.0000 - loss: 0.0047 - val_accuracy: 0.7778 - val_loss: 0.2256 - learning_rate: 0.0010
Epoch 3/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 711s 577ms/step - accuracy: 0.9831 - loss: 0.0422 - val_accuracy: 0.6531 - val_loss: 2.8257 - learning_rate: 0.0010
Epoch 4/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 1s 151us/step - accuracy: 0.9688 - loss: 0.0222 - val_accuracy: 0.7778 - val_loss: 0.5550 - learning_rate: 0.0010
Epoch 5/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step - accuracy: 0.9865 - loss: 0.0371
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 707s 574ms/step - accuracy: 0.9865 - loss: 0.0371 - val_accuracy: 0.6758 - val_loss: 1.96

Found 39375 images belonging to 2 classes.
Found 8617 images belonging to 2 classes.
Epoch 1/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 767s 618ms/step - accuracy: 0.9599 - loss: 0.1104 - val_accuracy: 0.6442 - val_loss: 2.2633 - learning_rate: 0.0010
Epoch 2/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 1s 271us/step - accuracy: 1.0000 - loss: 0.0047 - val_accuracy: 0.7778 - val_loss: 0.2256 - learning_rate: 0.0010
Epoch 3/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 711s 577ms/step - accuracy: 0.9831 - loss: 0.0422 - val_accuracy: 0.6531 - val_loss: 2.8257 - learning_rate: 0.0010
Epoch 4/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 1s 151us/step - accuracy: 0.9688 - loss: 0.0222 - val_accuracy: 0.7778 - val_loss: 0.5550 - learning_rate: 0.0010
Epoch 5/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step - accuracy: 0.9865 - loss: 0.0371
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 707s 574ms/step - accuracy: 0.9865 - loss: 0.0371 - val_accuracy: 0.6758 - val_loss: 1.9622 - learning_rate: 0.0010
Epoch 6/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 1s 163us/step - accuracy: 0.9688 - loss: 0.0132 - val_accuracy: 0.6667 - val_loss: 1.1997 - learning_rate: 1.0000e-04
Epoch 7/10
1230/1230 ━━━━━━━━━━━━━━━━━━━━ 700s 568ms/step - accuracy: 0.9913 - loss: 0.0249 - val_accuracy: 0.6759 - val_loss: 2.4280 - learning_rate: 1.0000e-04
Model compile time: 38 minutes, 27 seconds
  3/270 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - accuracy: 0.5990 - loss: 0.8744 
270/270 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - accuracy: 0.5885 - loss: 0.8938
Test Loss: 0.8996, Test Accuracy: 0.5831